In [1]:
import requests
import pandas as pd

LAT = 24.8607
LON = 67.0011
START = "2025-07-16"
END = "2025-08-03"

params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START,
    "end_date": END,
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "windspeed_10m",
        "winddirection_10m",
        "precipitation",
        "cloudcover",
        "surface_pressure"
    ]),
    "timezone": "auto"
}

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

response = requests.get(url, params=params).json()

# Print the full response to check the structure
print("🔍 Full API Response:")
print(response)

# Confirm if 'hourly' is present
if "hourly" in response:
    df = pd.DataFrame(response["hourly"])
    df["datetime"] = pd.to_datetime(df["time"])
    df.drop("time", axis=1, inplace=True)
    df.to_csv("raw_data.csv", index=False)
    print("✅ Saved raw_data.csv with shape:", df.shape)
else:
    print("❌ 'hourly' key not found in API response.")


🔍 Full API Response:
{'latitude': 24.875, 'longitude': 67.0, 'generationtime_ms': 1198.1436014175415, 'utc_offset_seconds': 18000, 'timezone': 'Asia/Karachi', 'timezone_abbreviation': 'GMT+5', 'elevation': 7.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'windspeed_10m': 'km/h', 'winddirection_10m': '°', 'precipitation': 'mm', 'cloudcover': '%', 'surface_pressure': 'hPa'}, 'hourly': {'time': ['2025-07-16T00:00', '2025-07-16T01:00', '2025-07-16T02:00', '2025-07-16T03:00', '2025-07-16T04:00', '2025-07-16T05:00', '2025-07-16T06:00', '2025-07-16T07:00', '2025-07-16T08:00', '2025-07-16T09:00', '2025-07-16T10:00', '2025-07-16T11:00', '2025-07-16T12:00', '2025-07-16T13:00', '2025-07-16T14:00', '2025-07-16T15:00', '2025-07-16T16:00', '2025-07-16T17:00', '2025-07-16T18:00', '2025-07-16T19:00', '2025-07-16T20:00', '2025-07-16T21:00', '2025-07-16T22:00', '2025-07-16T23:00', '2025-07-17T00:00', '2025-07-17T01:00', '2025-07-17T02:00', '2025-07-17T03:00',

In [2]:
df.head()

,temperature_2m,relative_humidity_2m,windspeed_10m,winddirection_10m,precipitation,cloudcover,surface_pressure,datetime
0,29.3,82,15.1,245,0.0,85,999.4,2025-07-16 00:00:00
1,29.2,83,13.4,246,0.0,97,999.2,2025-07-16 01:00:00
2,29.2,83,12.9,252,0.0,98,998.9,2025-07-16 02:00:00
3,29.1,83,13.6,253,0.0,100,998.7,2025-07-16 03:00:00
4,29.0,83,14.1,255,0.0,94,998.4,2025-07-16 04:00:00


In [3]:
import requests
import pandas as pd
from datetime import datetime

# Define coordinates (Karachi)
LAT = 24.8607
LON = 67.0011

# Set time range for historical data
HISTORICAL_START = "2025-06-25"
HISTORICAL_END = "2025-08-03"

# Set time range for forecast data (max 16 days ahead allowed)
FORECAST_START = "2025-07-19"
FORECAST_END = "2025-07-22"

# Common hourly weather variables
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "windspeed_10m",
    "winddirection_10m",
    "precipitation",
    "cloudcover",
    "surface_pressure"
]

def fetch_weather_data(api_url, start, end, lat, lon, variables, filename):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(variables),
        "timezone": "auto"
    }

    response = requests.get(api_url, params=params)
    data = response.json()

    if "hourly" in data:
        df = pd.DataFrame(data["hourly"])
        df["datetime"] = pd.to_datetime(df["time"])
        df.drop("time", axis=1, inplace=True)
        df.to_csv(filename, index=False)
        print(f"✅ Saved {filename} with shape: {df.shape}")
        return df
    else:
        print(f"❌ 'hourly' key not found in response from {api_url}")
        return pd.DataFrame()

# Fetch historical weather data
historical_api_url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
df_hist = fetch_weather_data(historical_api_url, HISTORICAL_START, HISTORICAL_END, LAT, LON, HOURLY_VARS, "historical_weather.csv")

# Fetch forecast weather data
forecast_api_url = "https://api.open-meteo.com/v1/forecast"
df_forecast = fetch_weather_data(forecast_api_url, FORECAST_START, FORECAST_END, LAT, LON, HOURLY_VARS, "forecast_weather.csv")


✅ Saved historical_weather.csv with shape: (960, 8)
✅ Saved forecast_weather.csv with shape: (96, 8)


In [4]:
import requests
import pandas as pd
from datetime import datetime

# AQICN API token and station ID
TOKEN = "7fd89733cef486ebaae225be3f9730a9630efe16"
STATION_ID = "A401143"
url = f"https://api.waqi.info/feed/{STATION_ID}/?token={TOKEN}"

# Step 1: Fetch pollutant data from AQICN
response = requests.get(url).json()

if response["status"] == "ok":
    data = response["data"]
    iaqi = data.get("iaqi", {})
    time_str = data["time"]["s"]  # Example: "2025-08-03 02:00:00"

    # Step 2: Convert to DataFrame
    aqi_data = {
        "datetime": pd.to_datetime(time_str),
        "aqi": data.get("aqi"),
        "pm25": iaqi.get("pm25", {}).get("v"),
        "pm10": iaqi.get("pm10", {}).get("v"),
        "no2": iaqi.get("no2", {}).get("v"),
        "co": iaqi.get("co", {}).get("v"),
        "so2": iaqi.get("so2", {}).get("v"),
        "o3": iaqi.get("o3", {}).get("v")
    }

    df_aqi = pd.DataFrame([aqi_data])

    print("✅ AQI Data:")
    print(df_aqi)

    # Step 3: Load historical weather data
    df_weather = pd.read_csv("historical_weather.csv")
    df_weather["datetime"] = pd.to_datetime(df_weather["datetime"])

    # Step 4: Merge on datetime
    df_merged = pd.merge(df_weather, df_aqi, on="datetime", how="left")

    print("✅ Merged Data (Weather + AQI):")
    print(df_merged.tail(3))

    # Optional: Save to file
    df_merged.to_csv("merged_weather_aqi.csv", index=False)
    print("✅ Saved merged_weather_aqi.csv with shape:", df_merged.shape)

else:
    print("❌ Failed to fetch AQI data:", response.get("data"))


✅ AQI Data:
             datetime  aqi  pm25  pm10   no2    co   so2    o3
0 2025-09-03 16:27:31   64    64    18  None  None  None  None
✅ Merged Data (Weather + AQI):
     temperature_2m  relative_humidity_2m  windspeed_10m  winddirection_10m  \
957            27.9                    84           14.8                229   
958            27.7                    84           14.3                231   
959            27.9                    86           12.0                224   

     precipitation  cloudcover  surface_pressure            datetime  aqi  \
957            0.0          73            1003.1 2025-08-03 21:00:00  NaN   
958            0.0          57            1003.8 2025-08-03 22:00:00  NaN   
959            0.0          44            1003.6 2025-08-03 23:00:00  NaN   

     pm25  pm10  no2   co  so2   o3  
957   NaN   NaN  NaN  NaN  NaN  NaN  
958   NaN   NaN  NaN  NaN  NaN  NaN  
959   NaN   NaN  NaN  NaN  NaN  NaN  
✅ Saved merged_weather_aqi.csv with shape: (960, 15)


In [5]:
import requests
import pandas as pd
import time
from datetime import datetime

# === Step 1: Set API Key ===
API_KEY = "6ba231e87114c1df16cde745209442d4"

# === Step 2: Set Location (Karachi) ===
LAT = 24.8607
LON = 67.0011

# === Step 3: Convert date range to UNIX timestamps ===
start_dt = datetime(2025, 6, 25)
end_dt = datetime(2025, 8, 3)

start_unix = int(time.mktime(start_dt.timetuple()))
end_unix = int(time.mktime(end_dt.timetuple()))

# === Step 4: API URL ===
url = f"http://api.openweathermap.org/data/2.5/air_pollution/history?lat={LAT}&lon={LON}&start={start_unix}&end={end_unix}&appid={API_KEY}"

# === Step 5: Fetch and Save Data ===
response = requests.get(url)
data = response.json()

# Parse data
aqi_data = []
for entry in data.get("list", []):
    dt = datetime.utcfromtimestamp(entry["dt"])
    aqi = entry["main"]["aqi"]
    components = entry["components"]
    aqi_data.append({
        "datetime": dt,
        "aqi": aqi,
        **components
    })

# Convert to DataFrame
df_aqi = pd.DataFrame(aqi_data)

# Save to CSV
df_aqi.to_csv("historical_aqi_openweather.csv", index=False)
print("✅ AQI data saved to 'historical_aqi_openweather.csv'")


✅ AQI data saved to 'historical_aqi_openweather.csv'


/tmp/ipython-input-3416754604.py:30: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  dt = datetime.utcfromtimestamp(entry["dt"])


In [6]:
import pandas as pd

# Load both datasets
weather_df = pd.read_csv("historical_weather.csv", parse_dates=["datetime"])
aqi_df = pd.read_csv("historical_aqi_openweather.csv", parse_dates=["datetime"])

# Round both datetime columns to nearest hour to ensure clean merge
weather_df["datetime"] = weather_df["datetime"].dt.round("H")
aqi_df["datetime"] = aqi_df["datetime"].dt.round("H")

# Merge on datetime
merged_df = pd.merge(weather_df, aqi_df, on="datetime", how="inner")

# Save the merged data
merged_df.to_csv("merged_weather_aqi.csv", index=False)
print(f"✅ Merged data saved to 'merged_weather_aqi.csv' with shape: {merged_df.shape}")


✅ Merged data saved to 'merged_weather_aqi.csv' with shape: (937, 17)


/tmp/ipython-input-2254269062.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  weather_df["datetime"] = weather_df["datetime"].dt.round("H")
/tmp/ipython-input-2254269062.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  aqi_df["datetime"] = aqi_df["datetime"].dt.round("H")


In [7]:
import pandas as pd

merged_df = pd.read_csv("merged_weather_aqi.csv")
display(merged_df.head())

,temperature_2m,relative_humidity_2m,windspeed_10m,winddirection_10m,precipitation,cloudcover,surface_pressure,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,29.6,90,7.9,210,0.0,100,995.3,2025-06-25 00:00:00,4,91.61,0.00,0.09,68.64,0.62,42.76,156.12,0.0
1,29.5,90,6.1,208,0.0,100,995.3,2025-06-25 01:00:00,4,93.51,0.00,0.09,71.71,0.63,43.56,157.92,0.0
2,29.4,90,4.4,215,0.0,100,995.2,2025-06-25 02:00:00,4,95.16,0.00,0.10,73.84,0.64,44.08,158.88,0.0
3,29.3,91,4.2,200,0.0,100,994.8,2025-06-25 03:00:00,4,96.33,0.00,0.10,75.18,0.65,44.24,159.31,0.0
4,29.1,91,4.6,198,0.0,100,994.3,2025-06-25 04:00:00,4,96.56,0.01,0.08,75.24,0.65,43.88,158.99,0.0


In [8]:
display(merged_df.tail())

,temperature_2m,relative_humidity_2m,windspeed_10m,winddirection_10m,precipitation,cloudcover,surface_pressure,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
932,28.4,81,15.8,227,0.0,73,1001.0,2025-08-02 20:00:00,3,74.71,0.0,0.08,44.56,0.34,20.64,78.27,0.0
933,28.1,83,14.8,231,0.0,75,1001.4,2025-08-02 21:00:00,3,73.44,0.0,0.08,44.63,0.33,20.70,78.08,0.0
934,27.9,84,13.6,238,0.0,58,1001.9,2025-08-02 22:00:00,3,72.48,0.0,0.08,44.84,0.32,20.23,74.42,0.0
935,28.2,85,11.5,238,0.0,60,1002.0,2025-08-02 23:00:00,3,72.34,0.0,0.08,45.30,0.31,19.36,68.04,0.0
936,28.0,86,11.8,243,0.0,81,1001.5,2025-08-03 00:00:00,3,72.61,0.0,0.08,45.98,0.30,18.69,62.79,0.0


In [9]:
# features.py
import pandas as pd

# Load the merged data
df = pd.read_csv("merged_weather_aqi.csv", parse_dates=["datetime"])

# Time-based features
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month

# AQI lag and rolling features
df["aqi_lag1"] = df["aqi"].shift(1)
df["aqi_lag2"] = df["aqi"].shift(2)
df["aqi_change_rate"] = df["aqi"] - df["aqi_lag1"]
df["aqi_rolling3"] = df["aqi"].rolling(window=3).mean()
df["aqi_rolling6"] = df["aqi"].rolling(window=6).mean()

# Drop rows with NaN created due to shift/rolling
df = df.dropna()

# Save processed features
df.to_csv("features.csv", index=False)
print("✅ Saved features.csv with shape:", df.shape)


✅ Saved features.csv with shape: (932, 25)


In [10]:
# train.py
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load features
df = pd.read_csv("features.csv")

# Define features and target
X = df.drop(columns=["datetime", "aqi"])
y = df["aqi"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate on training set
y_train_pred = model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Evaluate on test set
y_test_pred = model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)


print("✅ Training MSE:", train_mse)
print("✅ Training R²:", train_r2)
print("✅ Test MSE:", test_mse)
print("✅ Test R²:", test_r2)


# Save mode
import joblib
joblib.dump(model, "aqi_model.pkl")
print("✅ Model saved as aqi_model.pkl")

✅ Training MSE: 0.00073503355704698
✅ Training R²: 0.9985029686548214
✅ Test MSE: 0.0007946524064171128
✅ Test R²: 0.998353779620853
✅ Model saved as aqi_model.pkl
